# Daily Challenge: Stock Price Prediction with LSTM

In this notebook we build an end-to-end PyTorch LSTM pipeline to predict the **next day's closing price** from historical OHLCV (Open, High, Low, Close, Volume) data.

**Parts covered**
1. Install libraries  
2. Load & preprocess the dataset  
3. Prepare Dataset / DataLoader  
4. Define the LSTM model  
5. Train the model  
6. Evaluate (R²) & save artefacts  

## Step 1 — Install required libraries

In [ ]:
# Run once in Colab to install everything needed
# !pip install torch scikit-learn pandas numpy matplotlib joblib --quiet

import os, sys, math, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics       import r2_score
from pathlib import Path

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch version :', torch.__version__)
print('Device          :', DEVICE)

## Step 2 — Load and preprocess the dataset

**Dataset**  
Download the [Kaggle Stock Market Dataset](https://www.kaggle.com/datasets/jacksoncrow/stock-market-dataset) and place the CSV files inside `./data/stocks/`.  
Each file contains columns: `Date, Open, High, Low, Close, Volume`.

We concatenate all tickers, engineer features, create the **next-day Close** target, and normalise with `MinMaxScaler`.

In [ ]:
# ── Locate data ───────────────────────────────────────────────────────────────
for candidate in [
    Path('./data/stocks'),
    Path('/content/data/stocks'),
    Path('./stocks'),
]:
    if candidate.exists() and list(candidate.glob('*.csv')):
        STOCK_DIR = candidate
        break
else:
    raise FileNotFoundError(
        'No stock CSVs found. '
        'Place the Kaggle files in ./data/stocks/ and re-run.'
    )

csv_files = sorted(STOCK_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} ticker files in {STOCK_DIR}')
print('Examples:', [f.stem for f in csv_files[:5]])

In [ ]:
# ── Load all tickers into one DataFrame ──────────────────────────────────────
FEATURE_COLS = ['Open', 'High', 'Low', 'Close', 'Volume']
TARGET_COL   = 'Next_Close'      # next-day closing price (our label)

def load_ticker(path: Path) -> pd.DataFrame | None:
    """Load a single ticker CSV, add ticker column, drop bad rows."""
    try:
        df = pd.read_csv(path, parse_dates=['Date'])
        df = df.dropna(subset=FEATURE_COLS)
        df = df.sort_values('Date').reset_index(drop=True)

        # ── Drop unwanted columns (Adj Close, OpenInt, etc.) ──────────────
        df = df[['Date'] + FEATURE_COLS].copy()

        # ── Feature engineering ────────────────────────────────────────────
        df['Return_1d']   = df['Close'].pct_change()               # 1-day return
        df['High_Low_Pct']= (df['High'] - df['Low']) / df['Low']   # intraday range
        df['Vol_MA5']     = df['Volume'].rolling(5).mean()         # 5-day vol MA
        df['Close_MA5']   = df['Close'].rolling(5).mean()          # 5-day price MA
        df['Close_MA20']  = df['Close'].rolling(20).mean()         # 20-day price MA

        # ── Target: next day's close ───────────────────────────────────────
        df[TARGET_COL] = df['Close'].shift(-1)

        df['Ticker'] = path.stem
        df = df.dropna()   # remove NaN rows created by rolling & shift
        return df
    except Exception as e:
        print(f'  Skipping {path.name}: {e}')
        return None

frames = [load_ticker(f) for f in csv_files]
frames = [f for f in frames if f is not None and len(f) > 50]
df_all = pd.concat(frames, ignore_index=True)

print('Combined shape :', df_all.shape)
print('Tickers        :', df_all['Ticker'].nunique())
df_all.head()

In [ ]:
# ── Normalise ─────────────────────────────────────────────────────────────────
INPUT_COLS = ['Open','High','Low','Close','Volume',
              'Return_1d','High_Low_Pct','Vol_MA5','Close_MA5','Close_MA20']

X_raw = df_all[INPUT_COLS].values.astype(np.float32)
y_raw = df_all[TARGET_COL].values.astype(np.float32).reshape(-1, 1)

scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw).flatten()

print('X shape (normalised):', X_scaled.shape)
print('y shape (normalised):', y_scaled.shape)
print('X range: [{:.3f}, {:.3f}]'.format(X_scaled.min(), X_scaled.max()))
print('y range: [{:.3f}, {:.3f}]'.format(y_scaled.min(), y_scaled.max()))

## Step 3 — Prepare Dataset and DataLoader

We use a **sliding window** of `SEQ_LEN` consecutive rows as input to predict the next day's closing price.  
The split is **chronological**: train → validation → test (no leakage).

In [ ]:
SEQ_LEN    = 30     # look-back window: 30 trading days (~6 weeks)
BATCH_SIZE = 64

# ── Chronological split ───────────────────────────────────────────────────────
n = len(X_scaled)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X_scaled[:train_end],   y_scaled[:train_end]
X_val,   y_val   = X_scaled[train_end:val_end], y_scaled[train_end:val_end]
X_test,  y_test  = X_scaled[val_end:],     y_scaled[val_end:]

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')

In [ ]:
class StockDataset(Dataset):
    """
    Custom PyTorch Dataset that builds sliding-window sequences.

    Each item is:
        X : Tensor of shape (SEQ_LEN, n_features)  — input window
        y : scalar Tensor                           — next-day close (normalised)
    """

    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int):
        self.X       = torch.tensor(X, dtype=torch.float32)
        self.y       = torch.tensor(y, dtype=torch.float32)
        self.seq_len = seq_len

    def __len__(self) -> int:
        # Number of complete windows we can form
        return len(self.X) - self.seq_len

    def __getitem__(self, idx: int):
        x_window = self.X[idx : idx + self.seq_len]          # (SEQ_LEN, features)
        target   = self.y[idx + self.seq_len]                # scalar
        return x_window, target


train_ds = StockDataset(X_train, y_train, SEQ_LEN)
val_ds   = StockDataset(X_val,   y_val,   SEQ_LEN)
test_ds  = StockDataset(X_test,  y_test,  SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Verify shapes
sample_X, sample_y = next(iter(train_loader))
print('Batch X shape:', sample_X.shape)   # (BATCH_SIZE, SEQ_LEN, n_features)
print('Batch y shape:', sample_y.shape)   # (BATCH_SIZE,)

## Step 4 — Define the LSTM model

The architecture:
- **LSTM layer(s)** — capture sequential dependencies in price history  
- **Dropout** — regularise to avoid overfitting  
- **Linear (Dense) head** — map the LSTM's hidden state to a single price prediction

In [ ]:
class StockLSTM(nn.Module):
    """
    Two-layer stacked LSTM for next-day stock price regression.

    Args:
        input_size  : number of input features (10 in our case)
        hidden_size : LSTM hidden units per layer
        num_layers  : number of stacked LSTM layers
        dropout     : dropout probability between LSTM layers
        output_size : prediction horizon (1 = next day)
    """

    def __init__(
        self,
        input_size:  int = 10,
        hidden_size: int = 128,
        num_layers:  int = 2,
        dropout:     float = 0.2,
        output_size: int = 1,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # Stacked LSTM — dropout applied between layers (not after last layer)
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,        # input shape: (batch, seq, features)
            dropout     = dropout if num_layers > 1 else 0.0,
        )

        # Dropout before the dense head
        self.dropout = nn.Dropout(dropout)

        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (batch, seq_len, input_size)
        returns : (batch, output_size)
        """
        # Initialise hidden and cell states to zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)

        # LSTM forward pass
        out, _ = self.lstm(x, (h0, c0))  # out: (batch, seq_len, hidden_size)

        # Take the last time-step output
        last = out[:, -1, :]             # (batch, hidden_size)
        last = self.dropout(last)

        return self.fc(last).squeeze(-1) # (batch,)


# ── Instantiate ───────────────────────────────────────────────────────────────
N_FEATURES  = X_scaled.shape[1]   # 10
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
DROPOUT     = 0.2

model = StockLSTM(
    input_size  = N_FEATURES,
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
    output_size = 1,
).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal trainable parameters: {total_params:,}')

## Step 5 — Train the model

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS    = 30
LR        = 1e-3
PATIENCE  = 5        # early-stopping patience

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

print('Optimizer :', optimizer)
print('Loss      : MSELoss')

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    """One training OR validation pass. If optimizer is None, eval mode."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            preds = model(X_batch)           # (batch,)
            loss  = criterion(preds, y_batch)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(X_batch)

    return total_loss / len(loader.dataset)


# ── Training loop ─────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss  = float('inf')
patience_count = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss   = run_epoch(model, val_loader,   criterion, None,      DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        torch.save(model.state_dict(), '/home/claude/best_model.pt')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}')

print(f'\nBest validation MSE: {best_val_loss:.6f}')

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train MSE',      linewidth=1.5)
ax.plot(val_losses,   label='Validation MSE', linewidth=1.5)
ax.set_title('Training & Validation Loss (MSE)', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (normalised scale)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/home/claude/training_curves.png', dpi=120)
plt.show()

## Step 6 — Evaluate the model (R²) and save artefacts

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
model.load_state_dict(torch.load('/home/claude/best_model.pt', map_location=DEVICE))
model.eval()

# ── Collect test predictions ──────────────────────────────────────────────────
all_preds, all_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(DEVICE)).cpu().numpy()
        all_preds.append(preds)
        all_true.append(y_batch.numpy())

y_pred_scaled = np.concatenate(all_preds)
y_true_scaled = np.concatenate(all_true)

# ── R² on normalised scale ────────────────────────────────────────────────────
r2_normalised = r2_score(y_true_scaled, y_pred_scaled)
print(f'R² (normalised scale) : {r2_normalised:.4f}')

# ── Inverse-transform to original price scale ──────────────────────────────
y_pred_price = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true_price = scaler_y.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()

r2_price = r2_score(y_true_price, y_pred_price)
mae      = np.mean(np.abs(y_pred_price - y_true_price))
rmse     = np.sqrt(np.mean((y_pred_price - y_true_price) ** 2))

print(f'R² (original scale)   : {r2_price:.4f}')
print(f'MAE  (original scale) : {mae:.4f}')
print(f'RMSE (original scale) : {rmse:.4f}')

In [ ]:
# ── Plot predictions vs actuals (first 200 test steps) ────────────────────────
n_plot = min(200, len(y_true_price))

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Time-series overlay
axes[0].plot(y_true_price[:n_plot], label='Actual Close',    linewidth=1.2)
axes[0].plot(y_pred_price[:n_plot], label='Predicted Close', linewidth=1.2, alpha=0.8)
axes[0].set_title(f'Next-Day Close: Predicted vs Actual  (R²={r2_price:.4f})', fontsize=12)
axes[0].set_xlabel('Test step')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot — perfect model would lie on the diagonal
axes[1].scatter(y_true_price[:n_plot], y_pred_price[:n_plot],
                s=10, alpha=0.5, color='steelblue')
lo = min(y_true_price[:n_plot].min(), y_pred_price[:n_plot].min())
hi = max(y_true_price[:n_plot].max(), y_pred_price[:n_plot].max())
axes[1].plot([lo, hi], [lo, hi], 'r--', linewidth=1, label='Perfect fit')
axes[1].set_title('Actual vs Predicted (scatter)', fontsize=12)
axes[1].set_xlabel('Actual price')
axes[1].set_ylabel('Predicted price')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/predictions.png', dpi=120)
plt.show()

In [ ]:
# ── Save artefacts ────────────────────────────────────────────────────────────
os.makedirs('./saved_model', exist_ok=True)

# 1. Best model weights
torch.save(model.state_dict(), './saved_model/lstm_stock.pt')

# 2. Full model (architecture + weights) for easy reload
torch.save(model, './saved_model/lstm_stock_full.pt')

# 3. Scalers — required for inference on new raw data
joblib.dump(scaler_X, './saved_model/scaler_X.pkl')
joblib.dump(scaler_y, './saved_model/scaler_y.pkl')

# 4. Training metadata
import json
meta = {
    'input_cols'  : INPUT_COLS,
    'seq_len'     : SEQ_LEN,
    'hidden_size' : HIDDEN_SIZE,
    'num_layers'  : NUM_LAYERS,
    'dropout'     : DROPOUT,
    'best_val_mse': float(best_val_loss),
    'test_r2'     : float(r2_price),
    'test_mae'    : float(mae),
    'test_rmse'   : float(rmse),
}
with open('./saved_model/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to ./saved_model/')
print(json.dumps(meta, indent=2))

In [ ]:
# ── How to load and run inference on new data ─────────────────────────────────
# Load artefacts
loaded_model   = torch.load('./saved_model/lstm_stock_full.pt', map_location='cpu')
loaded_scaler_X = joblib.load('./saved_model/scaler_X.pkl')
loaded_scaler_y = joblib.load('./saved_model/scaler_y.pkl')

loaded_model.eval()

def predict_next_close(raw_window: np.ndarray) -> float:
    """
    Given a (SEQ_LEN, n_features) array of raw OHLCV+engineered values,
    return the predicted next-day closing price.
    """
    scaled  = loaded_scaler_X.transform(raw_window)               # normalise
    tensor  = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0)  # add batch dim
    with torch.no_grad():
        pred_scaled = loaded_model(tensor).item()
    price = loaded_scaler_y.inverse_transform([[pred_scaled]])[0, 0]
    return float(price)

# Quick smoke-test with the first test window
raw_test_window = X_raw[val_end : val_end + SEQ_LEN]          # shape (30, 10)
predicted_price = predict_next_close(raw_test_window)
true_price      = y_raw[val_end + SEQ_LEN, 0]
print(f'Smoke-test — Predicted: ${predicted_price:.2f}  |  True: ${true_price:.2f}')

## Summary

| Step | What we did |
|---|---|
| **Load** | Read all ticker CSVs, concatenated into one DataFrame |
| **Feature engineering** | 1-day return, intraday range, 5/20-day MAs, 5-day volume MA |
| **Target** | `Next_Close = Close.shift(-1)` — next day's closing price |
| **Normalisation** | `MinMaxScaler` fit on training split only (no leakage) |
| **Dataset** | Custom `StockDataset` with 30-day sliding windows |
| **Model** | 2-layer stacked LSTM (128 hidden units) + Dropout(0.2) + Linear |
| **Training** | Adam (lr=1e-3), MSE loss, gradient clipping, ReduceLROnPlateau, early stopping |
| **Evaluation** | R² score on original price scale + MAE + RMSE |
| **Saved** | Model weights, full model, scalers, metadata JSON |

**Key PyTorch concepts used**
- `nn.Module` — model base class  
- `nn.LSTM` — recurrent layer  
- `nn.Linear` — dense output head  
- `nn.Dropout` — regularisation  
- `nn.MSELoss` — regression loss  
- `torch.optim.Adam` — adaptive optimiser  
- `Dataset` / `DataLoader` — batching & shuffling  
- `torch.save` / `torch.load` — model persistence